# Data Handling & Visualization 3 — Missing values and basic cleaning

**Digital Foundation for Chemical Engineers**  \nDuration: 60 minutes

## Today’s goal
Detect missing and inconsistent values, choose a transparent cleaning action, and retain the distinction between raw and cleaned data.

## 1. Missing data are information (10 min)

A missing reading may mean a sensor was offline, a sample was not taken, or a value was not recorded. It is not automatically zero. In pandas, a missing numerical value is often displayed as `NaN` (not a number).

Before filling or dropping values, ask: *Why is it missing? Could the choice affect the engineering conclusion?*

In [3]:


raw_data = pd.read_csv('https://raw.githubusercontent.com/shreyashwasnik2006/Module_5_6_Both/refs/heads/main/data/industrial_process_100.csv')
# Simulated data-quality flags for this classroom exercise; retain the original CSV unchanged.
raw_data.loc[[17, 56], 'temperature_c'] = None
raw_data.loc[[33], 'flow_m3_h'] = None
raw_data.loc[[10], 'sample_status'] = ' ok '
raw_data.head(20)

,timestamp,batch_id,reactor,temperature_c,pressure_bar,flow_m3_h,conversion_percent,ph,operator,sample_status
0,2026-01-05 08:00,B1001,R1,73.0,4.65,11.70,87.5,6.70,Asha,OK
1,2026-01-05 09:00,B1001,R2,73.6,4.71,11.81,88.0,6.79,Bimal,OK
2,2026-01-05 10:00,B1001,R3,74.2,4.76,11.92,88.5,6.88,Chen,OK
3,2026-01-05 11:00,B1001,R1,74.9,4.82,12.03,89.1,6.97,Deepa,OK
4,2026-01-05 12:00,B1001,R2,75.5,4.87,12.14,89.6,7.06,Asha,OK
5,2026-01-05 13:00,B1001,R3,76.1,4.93,12.25,90.1,7.15,Bimal,OK
6,2026-01-05 14:00,B1001,R1,76.7,4.98,12.36,90.6,7.24,Chen,OK
7,2026-01-05 15:00,B1001,R2,77.3,5.04,12.47,91.1,6.70,Deepa,OK
8,2026-01-05 16:00,B1001,R3,78.0,5.09,12.58,91.7,6.79,Asha,OK
9,2026-01-05 17:00,B1001,R1,78.6,5.15,11.70,92.2,6.88,Bimal,OK


## 2. Finding missing and inconsistent values (12 min)

`isna()` identifies missing values; combining it with `sum()` counts them by column. For text fields, `value_counts()` helps reveal spelling/case/spacing inconsistencies.

In [4]:
print('Missing values by column:')
print(raw_data.isna().sum())

print('\nStatus values as recorded:')
print(raw_data['sample_status'].value_counts())

Missing values by column:
timestamp             0
batch_id              0
reactor               0
temperature_c         2
pressure_bar          0
flow_m3_h             1
conversion_percent    0
ph                    0
operator              0
sample_status         0
dtype: int64

Status values as recorded:
sample_status
OK           96
NOT TAKEN     3
 ok           1
Name: count, dtype: int64


**Try it (3 min):** Which variable has two missing values? Why would it be risky to replace missing readings with zero without thinking?

**Challenge:** Use `raw_data.isna().any(axis=1)` to show the rows that contain at least one missing value.

## 3. Cleaning text consistently (12 min)

Clean only what you understand. Here, leading/trailing spaces and different letter cases represent the same intended status. `str.strip()` removes edge spaces; `str.upper()` standardises case. We create a copy so the raw source remains available.

In [5]:
clean_data = raw_data.copy()
clean_data['sample_status'] = clean_data['sample_status'].str.strip().str.upper()
print(clean_data['sample_status'].value_counts())
clean_data

sample_status
OK           97
NOT TAKEN     3
Name: count, dtype: int64


,timestamp,batch_id,reactor,temperature_c,pressure_bar,flow_m3_h,conversion_percent,ph,operator,sample_status
0,2026-01-05 08:00,B1001,R1,73.0,4.65,11.70,87.5,6.70,Asha,OK
1,2026-01-05 09:00,B1001,R2,73.6,4.71,11.81,88.0,6.79,Bimal,OK
2,2026-01-05 10:00,B1001,R3,74.2,4.76,11.92,88.5,6.88,Chen,OK
3,2026-01-05 11:00,B1001,R1,74.9,4.82,12.03,89.1,6.97,Deepa,OK
4,2026-01-05 12:00,B1001,R2,75.5,4.87,12.14,89.6,7.06,Asha,OK
...,...,...,...,...,...,...,...,...,...,...
95,2026-01-09 07:00,B1010,R3,81.6,5.16,12.25,90.2,7.06,Deepa,OK
96,2026-01-09 08:00,B1010,R1,82.2,5.21,12.36,90.7,7.15,Asha,OK
97,2026-01-09 09:00,B1010,R2,82.8,5.27,12.47,91.2,7.24,Bimal,OK
98,2026-01-09 10:00,B1010,R3,83.5,5.32,12.58,91.7,6.70,Chen,OK


## 4. Choosing a missing-value treatment (15 min)

Common choices include:

- **Keep as missing:** appropriate when it must not be invented.
- **Drop a row:** only if the loss is acceptable and documented.
- **Fill from a justified value:** perhaps a mean for a simple classroom example.
- **Interpolate time-series readings:** only when process behaviour and sampling support it.

For serious industrial decisions, document the method and keep the original data.

In [6]:
# Demonstration only: fill missing temperature with the observed mean
mean_temperature_c = clean_data['temperature_c'].mean()
filled_temperature = clean_data['temperature_c'].fillna(mean_temperature_c)
print(f'Mean used: {mean_temperature_c:.2f} °C')
print(filled_temperature)

# A separate option: remove rows that lack flow data
flow_complete = clean_data.dropna(subset=['flow_m3_h'])
display(flow_complete)

Mean used: 79.14 °C
0     73.0
1     73.6
2     74.2
3     74.9
4     75.5
      ... 
95    81.6
96    82.2
97    82.8
98    83.5
99    84.1
Name: temperature_c, Length: 100, dtype: float64


,timestamp,batch_id,reactor,temperature_c,pressure_bar,flow_m3_h,conversion_percent,ph,operator,sample_status
0,2026-01-05 08:00,B1001,R1,73.0,4.65,11.70,87.5,6.70,Asha,OK
1,2026-01-05 09:00,B1001,R2,73.6,4.71,11.81,88.0,6.79,Bimal,OK
2,2026-01-05 10:00,B1001,R3,74.2,4.76,11.92,88.5,6.88,Chen,OK
3,2026-01-05 11:00,B1001,R1,74.9,4.82,12.03,89.1,6.97,Deepa,OK
4,2026-01-05 12:00,B1001,R2,75.5,4.87,12.14,89.6,7.06,Asha,OK
...,...,...,...,...,...,...,...,...,...,...
95,2026-01-09 07:00,B1010,R3,81.6,5.16,12.25,90.2,7.06,Deepa,OK
96,2026-01-09 08:00,B1010,R1,82.2,5.21,12.36,90.7,7.15,Asha,OK
97,2026-01-09 09:00,B1010,R2,82.8,5.27,12.47,91.2,7.24,Bimal,OK
98,2026-01-09 10:00,B1010,R3,83.5,5.32,12.58,91.7,6.70,Chen,OK


**Pair task (8 min):** Add a `pressure_bar` column containing one `None` and one suspicious text value such as `'unknown'`. Decide, in words first, how you would handle each problem.

**Challenge:** Convert numeric-looking text with `pd.to_numeric(..., errors='coerce')`, then count the newly missing values.

In [11]:
# Your cleaning decision and code
raw_data.loc[[3], 'pressure_bar'] = None




## Exit check

You can locate `NaN`, standardise text, make a deliberate cleaning choice, and explain why raw data should not be overwritten. Next: sorting, filtering, and simple summaries.